In [1]:
from datasets import load_dataset

print("Downloading ChatAlpaca-20K...")
chatalpaca = load_dataset("robinsmits/ChatAlpaca-20K", split="train")
print(f"Train examples: {len(chatalpaca)}")
print(chatalpaca[0])

README.md: 0.00B [00:00, ?B/s]

C:\Users\MIND & MATTER\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MIND & MATTER\.cache\huggingface\hub\datasets--robinsmits--ChatAlpaca-20K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/original-00000-of-00001.parquet:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/33.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/684k [00:00<?, ?B/s]

Generating original split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/19600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

Train examples: 19600
{'id': 0, 'messages': [{'content': 'Find the product of the numbers: 5 and 8', 'role': 'user'}, {'content': 'The product of 5 and 8 is 40.', 'role': 'assistant'}, {'content': 'What is the sum of the numbers 6 and 12?', 'role': 'user'}, {'content': 'The sum of the numbers 6 and 12 is 18.', 'role': 'assistant'}, {'content': 'Can you tell me the quotient of 20 and 5?', 'role': 'user'}, {'content': 'Yes, the quotient of 20 and 5 is 4.', 'role': 'assistant'}, {'content': 'What is the difference between 25 and 13?', 'role': 'user'}, {'content': 'The difference between 25 and 13 is 12.', 'role': 'assistant'}, {'content': 'What is the square of 9?', 'role': 'user'}, {'content': 'The square of 9 is 81.', 'role': 'assistant'}, {'content': 'What is the cube of 6?', 'role': 'user'}, {'content': 'The cube of 6 is 216.', 'role': 'assistant'}]}


In [2]:
chatalpaca_test = load_dataset("robinsmits/ChatAlpaca-20K", split="test")
print(f"Test examples: {len(chatalpaca_test)}")


Test examples: 400


In [3]:
import os

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

chatalpaca.to_json(os.path.join(RAW_DIR, "chatalpaca_train_raw.jsonl"))
chatalpaca_test.to_json(os.path.join(RAW_DIR, "chatalpaca_test_raw.jsonl"))
print("Saved raw files.")

Creating json from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved raw files.


In [6]:
import json

INPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\chatalpaca_train_raw.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_filtered.jsonl"

FILLER_PATTERNS = [
    "as an ai language model",
    "as an ai, i",
    "i don't have personal preferences",
    "i do not have personal preferences",
]

def has_filler(text):
    lowered = text.lower()
    return any(p in lowered for p in FILLER_PATTERNS)

kept, dropped = 0, 0
with open(INPUT, encoding="utf-8") as fin, open(OUTPUT, "w", encoding="utf-8") as fout:
    for line in fin:
        d = json.loads(line)
        msgs = d["messages"]
        if any(has_filler(m["content"]) for m in msgs if m["role"] == "assistant"):
            dropped += 1
            continue
        fout.write(json.dumps(d) + "\n")
        kept += 1

print(f"Kept: {kept}, Dropped: {dropped}")

Kept: 16057, Dropped: 3543


In [8]:
import json

lines = open(r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_filtered.jsonl", encoding="utf-8").readlines()
over_2000 = 0
total_msgs = 0
for l in lines:
    d = json.loads(l)
    for m in d["messages"]:
        total_msgs += 1
        if len(m["content"]) > 2000:
            over_2000 += 1

print(f"Messages over 2000 chars: {over_2000} / {total_msgs}")

Messages over 2000 chars: 317 / 140812


In [9]:
import json

INPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_filtered.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_final.jsonl"
MAX_CHARS = 2000

kept, dropped = 0, 0
with open(INPUT, encoding="utf-8") as fin, open(OUTPUT, "w", encoding="utf-8") as fout:
    for line in fin:
        d = json.loads(line)
        if any(len(m["content"]) > MAX_CHARS for m in d["messages"]):
            dropped += 1
            continue
        fout.write(json.dumps(d) + "\n")
        kept += 1

print(f"Kept: {kept}, Dropped: {dropped}")

Kept: 15787, Dropped: 270


In [10]:
import json

INPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_final.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_converted.jsonl"

def build_examples(messages):
    examples = []
    history = []  # list of (role, content) pairs prior to current instruction

    for i in range(len(messages) - 1):
        if messages[i]["role"] != "user" or messages[i+1]["role"] != "assistant":
            continue

        instruction = messages[i]["content"]
        response = messages[i+1]["content"]

        if history:
            context_lines = []
            for role, content in history:
                speaker = "User" if role == "user" else "Assistant"
                context_lines.append(f"{speaker}: {content}")
            context_block = "\n".join(context_lines)
            text = f"### Context:\n{context_block}\n\n### Instruction:\n{instruction}\n\n### Response:\n{response}"
        else:
            text = f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

        examples.append({"text": text})

        # extend history with this exchange for the next iteration
        history.append(("user", instruction))
        history.append(("assistant", response))

    return examples

total_examples = 0
with open(INPUT, encoding="utf-8") as fin, open(OUTPUT, "w", encoding="utf-8") as fout:
    for line in fin:
        d = json.loads(line)
        examples = build_examples(d["messages"])
        for ex in examples:
            fout.write(json.dumps(ex) + "\n")
            total_examples += 1

print(f"Total training examples generated: {total_examples}")

Total training examples generated: 69220


In [11]:
import json

FILLER_PATTERNS = [
    "as an ai language model",
    "as an ai, i",
    "i don't have personal preferences",
    "i do not have personal preferences",
]

def has_filler(text):
    lowered = text.lower()
    return any(p in lowered for p in FILLER_PATTERNS)

INPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\chatalpaca_test_raw.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_val_final.jsonl"
MAX_CHARS = 2000

kept, dropped = 0, 0
with open(INPUT, encoding="utf-8") as fin, open(OUTPUT, "w", encoding="utf-8") as fout:
    for line in fin:
        d = json.loads(line)
        msgs = d["messages"]
        if any(has_filler(m["content"]) for m in msgs if m["role"] == "assistant"):
            dropped += 1
            continue
        if any(len(m["content"]) > MAX_CHARS for m in msgs):
            dropped += 1
            continue
        fout.write(json.dumps(d) + "\n")
        kept += 1

print(f"Kept: {kept}, Dropped: {dropped}")

Kept: 333, Dropped: 67


In [12]:
import json

INPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_val_final.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\chatalpaca_val_test_converted.jsonl"

def build_examples(messages):
    examples = []
    history = []

    for i in range(len(messages) - 1):
        if messages[i]["role"] != "user" or messages[i+1]["role"] != "assistant":
            continue

        instruction = messages[i]["content"]
        response = messages[i+1]["content"]

        if history:
            context_lines = []
            for role, content in history:
                speaker = "User" if role == "user" else "Assistant"
                context_lines.append(f"{speaker}: {content}")
            context_block = "\n".join(context_lines)
            text = f"### Context:\n{context_block}\n\n### Instruction:\n{instruction}\n\n### Response:\n{response}"
        else:
            text = f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

        examples.append({"text": text})

        history.append(("user", instruction))
        history.append(("assistant", response))

    return examples

total_examples = 0
with open(INPUT, encoding="utf-8") as fin, open(OUTPUT, "w", encoding="utf-8") as fout:
    for line in fin:
        d = json.loads(line)
        examples = build_examples(d["messages"])
        for ex in examples:
            fout.write(json.dumps(ex) + "\n")
            total_examples += 1

print(f"Total val examples generated: {total_examples}")

Total val examples generated: 1420
